# A first look at gas-turbine emissions data

**MECE 4520 · Fall 2026 · asynchronous foundations**

<a target="_blank" href="https://colab.research.google.com/github/changyaochen/MECE4520/blob/master/site/foundations/02-gas-turbine-data.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"/>
</a>

This notebook introduces our primary recurring dataset. Before fitting a model, engineers should understand what was measured, what each observation represents, and which patterns are plausible.

**Learning objectives**

- Inspect a dataset for size, variable types, and missing values.
- Summarize measurements with appropriate descriptive statistics.
- Use plots to ask physical questions about a system.
- Distinguish an observed association from a causal conclusion.

## 1. Load the data

The UCI dataset contains 36,733 hourly observations from a gas turbine, measured between 2011 and 2015. Its rows are chronologically ordered, although individual timestamps are not supplied.

In [ ]:
%pip -q install ucimlrepo

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from ucimlrepo import fetch_ucirepo

plt.style.use("seaborn-v0_8-whitegrid")

gas_turbine = fetch_ucirepo(id=551)
data = pd.concat([gas_turbine.data.features, gas_turbine.data.targets], axis=1)
data.head()

## 2. What is in the table?

Each row is an hourly operating observation. Variables include ambient conditions, pressure and temperature measurements, turbine energy yield, and the emissions outcomes CO and NOx.

In [ ]:
print(f"Observations: {data.shape[0]:,}")
print(f"Variables: {data.shape[1]}")

summary = pd.DataFrame({
    "dtype": data.dtypes.astype(str),
    "missing": data.isna().sum(),
    "unique values": data.nunique(),
})
summary

There are no missing values in this version of the dataset. That is convenient for teaching, but it should not be treated as normal: checking missingness is an essential first step in every real analysis.

## 3. Summarize key measurements

The mean alone is rarely enough. The standard deviation and quantiles describe how much a measurement varies across observed operating conditions.

In [ ]:
key_variables = ["AT", "AP", "AH", "TIT", "TEY", "CO", "NOX"]
data[key_variables].describe().T[["mean", "std", "min", "25%", "50%", "75%", "max"]]

## 4. Distribution of an emissions outcome

A histogram shows how frequently a range of values occurs. The distribution of NOx is not perfectly symmetric, so reporting only a mean can hide useful information about high-emission observations.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.hist(data["NOX"], bins=50, color="#4a90c2", edgecolor="white")
ax.axvline(data["NOX"].mean(), color="#c43c35", linestyle="--", label="mean")
ax.axvline(data["NOX"].median(), color="#222222", linestyle=":", label="median")
ax.set(xlabel="NOx emissions (mg/m³)", ylabel="Number of hourly observations", title="Distribution of NOx emissions")
ax.legend()
plt.show()

## 5. Explore associations

A correlation table is a compact overview of linear associations. It is useful for asking questions, not for deciding that one variable causes another. Sensors can be related because they respond to the same operating state.

In [ ]:
correlations = data[key_variables].corr(numeric_only=True)
correlations.round(2)

In [ ]:
plot_data = data.sample(n=3_000, random_state=4520)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2), constrained_layout=True)
axes[0].scatter(plot_data["TIT"], plot_data["NOX"], alpha=0.2, s=10)
axes[0].set(xlabel="Turbine inlet temperature, TIT (°C)", ylabel="NOx (mg/m³)", title="Temperature and NOx")

axes[1].scatter(plot_data["TEY"], plot_data["CO"], alpha=0.2, s=10, color="#c43c35")
axes[1].set(xlabel="Turbine energy yield, TEY (MWh)", ylabel="CO (mg/m³)", title="Energy yield and CO")

plt.show()

## Check-in

1. Which variable has the most obvious association with NOx in the plots or correlation table?
2. What operating conditions might create a relationship between two sensors even if neither directly causes the other?
3. Why might chronology matter when we later split these observations into training and test sets?

Keep your answers in your own notes. The next notebook uses the same measurements to study variability and statistical evidence.